<a href="https://colab.research.google.com/github/Gunjannn07/FlyRank1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gunjannn07/FlyRank1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a **Decision Tree** for this modeling lane.

A Decision Tree fits the CTR opportunity lane because it can learn simple rules from page-level signals and is easy to interpret. It also allows me to see which features are most useful when deciding which pages should be reviewed.

I chose a Decision Tree rather than a more complex model because the goal is to build an honest model that can be compared with the Week-4 baseline, not to reward complexity alone.



In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
from sklearn.tree import DecisionTreeClassifier

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use the **80-20 split** where 80% of the data is for training and the 20% for testing.
This prevents pages from the same client appearing in both the training and test sets. It gives a more honest estimate of how the model performs on clients it did not see during training.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [12]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Training rows:", len(train))
print("Testing rows:", len(test))

print("Training clients:", train["client_id"].nunique())
print("Testing clients:", test["client_id"].nunique())

print(
    "Shared clients:",
    len(set(train["client_id"]) & set(test["client_id"]))
)

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train a Decision Tree using the training clients and evaluate it on the held-out clients.

The model will use the same CTR-opportunity signals used by the Week-4 baseline. I will compare the model ranking with the Week-4 baseline ranking on the same test set.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [14]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Create position buckets
position_bins = [0, 3, 10, 20, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21+"]
train["position_bucket"] = pd.cut(
    train["avg_position"],
    bins=position_bins, labels=position_labels, include_lowest=True
)

test["position_bucket"] = pd.cut(
    test["avg_position"],
    bins=position_bins,labels=position_labels, include_lowest=True
)

# Features for the Decision Tree
features = [ "search_volume", "avg_position","engagement_rate","scroll_rate", "ai_traffic_pct"
]

X_train = train[features].fillna(0)
X_test = test[features].fillna(0)

y_train = train["ctr"].fillna(train["ctr"].median())
y_test = test["ctr"].fillna(train["ctr"].median())

# Train Decision Tree
model = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

# Model predictions
test["model_ctr"] = model.predict(X_test)

baseline_ctr = (
    train.groupby("position_bucket", observed=True)["ctr"]
    .mean()
)
test["baseline_ctr"] = (
    test["position_bucket"]
    .map(baseline_ctr)
    .astype(float)
)

# Calculate errors
model_mae = mean_absolute_error(
    y_test,
    test["model_ctr"]
)

baseline_mae = mean_absolute_error(
    y_test,
    test["baseline_ctr"].fillna(y_train.mean())
)

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Decision Tree"],
    "MAE": [baseline_mae, model_mae]})
display(comparison)

,Method,MAE
0,Week-4 Baseline,0.545452
1,Decision Tree,0.607709


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The largest Decision Tree errors occur on pages with unusually high observed CTR. For example, several pages have CTR values above 20%, while the model predicts values below 3%. This shows that the model does not capture some of these high-CTR cases well.

The Decision Tree also performed worse than the Week-4 baseline, with an MAE of 0.607709 compared with 0.545452 for the baseline. Therefore, the model did not improve on the simpler baseline for this test set.



In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [16]:
# Calculate absolute errors
test["model_error"] = abs(test["ctr"] - test["model_ctr"])
test["baseline_error"] = abs(test["ctr"] - test["baseline_ctr"])

errors = test.sort_values(
    "model_error",
    ascending=False
)

display(
    errors[
        [
            "content_id","ctr","model_ctr","baseline_ctr","model_error","baseline_error","avg_position","search_volume"
        ]
    ].head(10)
)
# Feature importance
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)
display(importance)

,content_id,ctr,model_ctr,baseline_ctr,model_error,baseline_error,avg_position,search_volume
4500,content_f2eeaeefe9f1,33.33,0.019832,1.649935,33.310168,31.680065,1.7,10.0
22847,content_77c30aa8260f,33.33,0.443459,0.229817,32.886541,33.100183,25.0,0.0
9129,content_3db3c9053d5c,25.00,2.310000,1.649935,22.690000,23.350065,1.3,0.0
5544,content_9ba30bec775e,20.00,0.294712,0.341592,19.705288,19.658408,13.6,110.0
18159,content_6daf2739da2c,16.67,0.294712,0.341592,16.375288,16.328408,11.3,20.0
11387,content_dd9ab01faf99,16.67,0.294712,0.733742,16.375288,15.936258,6.0,10.0
11802,content_c90cac472cbc,0.00,12.500000,0.229817,12.500000,0.229817,77.0,10.0
1697,content_06521c9216a0,14.29,2.310000,1.649935,11.980000,12.640065,0.6,0.0
13881,content_6fd907bb1711,11.63,0.443459,0.733742,11.186541,10.896258,7.9,0.0
7847,content_4a8ff0b56049,11.11,0.294712,0.733742,10.815288,10.376258,6.6,10.0


,feature,importance
1,avg_position,0.485496
3,scroll_rate,0.215659
0,search_volume,0.191413
2,engagement_rate,0.107432
4,ai_traffic_pct,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.